In [68]:
import numpy as np
import pandas as pd

# Load and Clean Excel Sheets as DFs:

In [69]:
CO2_df = pd.read_excel("Emissions2024.xlsx", sheet_name='CO2', header=1, skipfooter=2).query("`Plant Code`!=99999")
SO2_df = pd.read_excel("Emissions2024.xlsx", sheet_name='SO2', header=1, skipfooter=2).query("`Plant Code`!=99999")
NOx_df = pd.read_excel("Emissions2024.xlsx", sheet_name='NOx', header=1, skipfooter=2).query("`Plant Code`!=99999")

# Create a unique key column for each DF:

In [70]:
CO2_df['Plant Code'] = CO2_df['Plant Code'].astype('str')
CO2_df['Sector Code'] = CO2_df['Sector Code'].astype('str')
CO2_df['Plant_Sec_PM_FC'] = CO2_df['Plant Code'] + '_' + CO2_df['Sector Code'] + '_' + CO2_df['Prime Mover'] + '_' + CO2_df['Fuel Code']

In [71]:
SO2_df['Plant Code'] = SO2_df['Plant Code'].astype('str')
SO2_df['Sector Code'] = SO2_df['Sector Code'].astype('str')
SO2_df['Plant_Sec_PM_FC'] = SO2_df['Plant Code'] + '_' + SO2_df['Sector Code'] + '_' + SO2_df['Prime Mover'] + '_' + SO2_df['Fuel Code']

In [72]:
NOx_df['Plant Code'] = NOx_df['Plant Code'].astype('str')
NOx_df['Sector Code'] = NOx_df['Sector Code'].astype('str')
NOx_df['Plant_Sec_PM_FC'] = NOx_df['Plant Code'] + '_' + NOx_df['Sector Code'] + '_' + NOx_df['Prime Mover'] + '_' + NOx_df['Fuel Code']

# Inner Merge Emissions Outputs into one DF:

In [73]:
result = pd.merge(
    CO2_df, 
    SO2_df[['Plant_Sec_PM_FC', 'EIA Model Estimates of SO2 Emissions (Tons)']], 
    on='Plant_Sec_PM_FC', 
    how='inner'
)

In [75]:
total_pollution_df = pd.merge(
    result, 
    NOx_df[['Plant_Sec_PM_FC', 'EIA Model Estimates of NOx Emissions (Tons)']], 
    on='Plant_Sec_PM_FC', 
    how='inner'
)

In [104]:
total_pollution_df.head()

,Plant Code,Plant Name,State,Sector Group,Sector Code,Prime Mover,Fuel Code,Aggregated Fuel Group,Generation (kWh),Useful Thermal Output (MMBtu),...,Tons of CO2 Emissions,Metric Tonnes of CO2 Emissions,NERC Region,Balancing Authority Code,Balancing Authority Name,EIA Balancing Authority Region,Plant_Sec_PM_FC,EIA Model Estimates of SO2 Emissions (Tons),EIA Model Estimates of NOx Emissions (Tons),Fuel Consumption for Useful Thermal Output (MMBtu)
0,1,Sand Point,AK,ELECTRIC POWER,1,IC,DFO,PET,3616000,0,...,3196,2899,NaN,NaN,Balancing Authority Not Reported,Alaska,1_1_IC_DFO,5.006497,58.779979,0
1,3,Barry,AL,ELECTRIC POWER,1,CA,NG,GAS,4519611000,0,...,188829,171303,SERC,SOCO,"Southern Company Services, Inc. - Trans",Southeast,3_1_CA_NG,0.951405,354.67097,0
2,3,Barry,AL,ELECTRIC POWER,1,CT,NG,GAS,8653458000,0,...,5032786,4565672,SERC,SOCO,"Southern Company Services, Inc. - Trans",Southeast,3_1_CT_NG,25.354901,11832.287,0
3,3,Barry,AL,ELECTRIC POWER,1,ST,BIT,COAL,697528563,0,...,802806,728294,SERC,SOCO,"Southern Company Services, Inc. - Trans",Southeast,3_1_ST_BIT,273.69221,217.5921,0
4,3,Barry,AL,ELECTRIC POWER,1,ST,NG,GAS,185271437,0,...,156709,142164,SERC,SOCO,"Southern Company Services, Inc. - Trans",Southeast,3_1_ST_NG,0.67211,67.171504,0


# Remove irrelevant columns?

In [78]:
nan_cols = total_pollution_df.columns[total_pollution_df.isna().any()].tolist()
print(nan_cols)

['NERC Region', 'Balancing Authority Code']


In [90]:
total_pollution_df['Fuel Consumption for Useful Thermal Output (MMBtu)'] = total_pollution_df['\n Fuel Consumption for Useful Thermal Output (MMBtu)']

In [91]:
total_pollution_df.columns

Index(['Plant Code', 'Plant Name', 'State', 'Sector Group', 'Sector Code',
       'Prime Mover', 'Fuel Code', 'Aggregated Fuel Group', 'Generation (kWh)',
       'Useful Thermal Output (MMBtu)', 'Total Fuel Consumption (MMBtu)',
       'Fuel Consumption for Electric Generation (MMBtu)',
       '\n Fuel Consumption for Useful Thermal Output (MMBtu)',
       'Quantity of Fuel Consumed', 'Fuel Units', 'Tons of CO2 Emissions',
       'Metric Tonnes of CO2 Emissions', 'NERC Region',
       'Balancing Authority Code', 'Balancing Authority Name',
       'EIA Balancing Authority Region', 'Plant_Sec_PM_FC',
       'EIA Model Estimates of SO2 Emissions (Tons)',
       'EIA Model Estimates of NOx Emissions (Tons)',
       'Fuel Consumption for Useful Thermal Output (MMBtu)'],
      dtype='str')

In [95]:
cols = ['Plant_Sec_PM_FC', 'Plant Code', 'Plant Name', 'State', 'Sector Group', 'Sector Code',
       'Prime Mover', 'Fuel Code', 'Aggregated Fuel Group', 'Generation (kWh)',
       'Useful Thermal Output (MMBtu)', 'Total Fuel Consumption (MMBtu)',
       'Fuel Consumption for Electric Generation (MMBtu)',
       'Fuel Consumption for Useful Thermal Output (MMBtu)',
       'Quantity of Fuel Consumed', 'Fuel Units',
       'EIA Balancing Authority Region', 'Tons of CO2 Emissions',
       'EIA Model Estimates of SO2 Emissions (Tons)',
       'EIA Model Estimates of NOx Emissions (Tons)']

In [96]:
tp_clean = total_pollution_df[cols]
tp_clean

,Plant_Sec_PM_FC,Plant Code,Plant Name,State,Sector Group,Sector Code,Prime Mover,Fuel Code,Aggregated Fuel Group,Generation (kWh),Useful Thermal Output (MMBtu),Total Fuel Consumption (MMBtu),Fuel Consumption for Electric Generation (MMBtu),Fuel Consumption for Useful Thermal Output (MMBtu),Quantity of Fuel Consumed,Fuel Units,EIA Balancing Authority Region,Tons of CO2 Emissions,EIA Model Estimates of SO2 Emissions (Tons),EIA Model Estimates of NOx Emissions (Tons)
0,1_1_IC_DFO,1,Sand Point,AK,ELECTRIC POWER,1,IC,DFO,PET,3616000,0,39104,39104,0,6307,Barrels,Alaska,3196,5.006497,58.779979
1,3_1_CA_NG,3,Barry,AL,ELECTRIC POWER,1,CA,NG,GAS,4519611000,0,3237638,3237638,0,3171351,Mcf,Southeast,188829,0.951405,354.67097
2,3_1_CT_NG,3,Barry,AL,ELECTRIC POWER,1,CT,NG,GAS,8653458000,0,86291289,86291289,0,84516338,Mcf,Southeast,5032786,25.354901,11832.287
3,3_1_ST_BIT,3,Barry,AL,ELECTRIC POWER,1,ST,BIT,COAL,697528563,0,7810960,7810960,0,395622,Tons,Southeast,802806,273.69221,217.5921
4,3_1_ST_NG,3,Barry,AL,ELECTRIC POWER,1,ST,NG,GAS,185271437,0,2686899,2686899,0,2630417,Mcf,Southeast,156709,0.67211,67.171504
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4877,68733_2_IC_NG,68733,HEB00798,TX,ELECTRIC POWER,2,IC,NG,GAS,136000,0,2294,2294,0,1911,Mcf,"Electric Reliability Council of Texas, Inc.",134,0.000573,2.644824
4878,68734_2_IC_NG,68734,HEB00801,TX,ELECTRIC POWER,2,IC,NG,GAS,146000,0,2178,2178,0,1815,Mcf,"Electric Reliability Council of Texas, Inc.",127,0.000544,2.51196
4879,68735_2_IC_NG,68735,HEBTEMDistCenter1B,TX,ELECTRIC POWER,2,IC,NG,GAS,309000,0,5264,5264,0,4387,Mcf,"Electric Reliability Council of Texas, Inc.",307,0.001316,6.071608
4880,68736_2_IC_NG,68736,HEBTEMDistCenter2A,TX,ELECTRIC POWER,2,IC,NG,GAS,309000,0,5264,5264,0,4387,Mcf,"Electric Reliability Council of Texas, Inc.",307,0.001316,6.071608


# Might want to remove irrelevant columns in first 3 dfs